# Teste isolado — Diário Oficial de SP (busca por termo, API JSON)

Fonte candidata: **SPI - Secretaria de Parcerias em Investimentos** (SP),
via busca por termo no Diário Oficial do Estado. Notebook **descartável**
(Fase 1) — sem dispatcher, sem gravar nada.

## Confirmado antes de assumir

Site é uma SPA Next.js — HTML puro só traz o "shell" da página, sem
conteúdo (mesmo sintoma do Brazil Journal). Mas, diferente do Brazil
Journal, achamos a API JSON por trás via DevTools (Network > Fetch/XHR),
então não precisa de Selenium.

**Duas chamadas, confirmadas manualmente**:
1. Busca: `GET /v2/advanced-search/publications?Terms[0]={termo}&FromDate=...&ToDate=...&PageNumber=...&PageSize=...&SortField=Date`
   — devolve lista paginada (`items`, `totalPages`, `hasNextPage`), cada
   item com `slug`, `title`, `date`, `hierarchy`, `excerpt`.
2. Detalhe: `GET /v2/publications/{slug}` — devolve o registro completo,
   incluindo `content` (HTML com entidades, precisa decodificar + tirar tag).

**Achado**: a busca é por *menção ao termo*, não só publicação *da*
secretaria — um resultado de teste pra "spi" saiu sendo na verdade um
extrato de contrato da ARTESP que cita a SPI como poder concedente. Vale
considerar isso na hora de avaliar relevância.

In [0]:
%pip install --quiet httpx beautifulsoup4 lxml
dbutils.library.restartPython()


In [0]:
import time
import random
from datetime import datetime, timedelta
from typing import Optional

import httpx
from bs4 import BeautifulSoup


In [0]:
API_BASE = "https://do-api-web-search.doe.sp.gov.br/v2"

TERMOS = ["spi"]

HOJE = datetime.today()
FROM_DATE = (HOJE - timedelta(days=31)).strftime("%Y-%-m-%-d")
TO_DATE = HOJE.strftime("%Y-%-m-%-d")

PAGE_SIZE = 20
HTTP_TIMEOUT = 30

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)


## Etapa 1 — Buscar publicações por termo (todas as páginas)

In [0]:
def buscar_publicacoes(termo: str, from_date: str, to_date: str, page_size: int = PAGE_SIZE) -> list[dict]:
    itens = []
    pagina = 1

    while True:
        params = {
            "Terms[0]": termo,
            "FromDate": from_date,
            "ToDate": to_date,
            "PageNumber": pagina,
            "PageSize": page_size,
            "SortField": "Date",
        }
        resp = httpx.get(
            f"{API_BASE}/advanced-search/publications",
            params=params,
            headers={"User-Agent": USER_AGENT},
            timeout=HTTP_TIMEOUT,
        )
        if resp.status_code != 200:
            print(f"  [busca] página {pagina} -> status {resp.status_code}, parando.")
            break

        dados = resp.json()
        pagina_itens = dados.get("items", [])
        itens.extend(pagina_itens)

        print(f"  [busca] página {pagina}/{dados.get('totalPages', '?')}: {len(pagina_itens)} itens.")

        if not dados.get("hasNextPage"):
            break
        pagina += 1
        time.sleep(random.uniform(0.3, 0.8))

    return itens


todos_itens = []
for termo in TERMOS:
    print(f"Termo: {termo!r}")
    itens_termo = buscar_publicacoes(termo, FROM_DATE, TO_DATE)
    for item in itens_termo:
        item["_termo_busca"] = termo
    todos_itens.extend(itens_termo)

print(f"\nTotal de itens encontrados: {len(todos_itens)}")

sem_data = [i for i in todos_itens if not i.get("date")]
sem_slug = [i for i in todos_itens if not i.get("slug")]
print(f"Sem data: {len(sem_data)}")
print(f"Sem slug: {len(sem_slug)}")


## Etapa 2 — Buscar o detalhe de uma amostra, decodificar o `content`

In [0]:
def obter_detalhe(slug: str) -> Optional[dict]:
    resp = httpx.get(
        f"{API_BASE}/publications/{slug}",
        headers={"User-Agent": USER_AGENT},
        timeout=HTTP_TIMEOUT,
    )
    if resp.status_code != 200:
        print(f"    -> status {resp.status_code}")
        return None
    return resp.json()


def limpar_html_content(html_bruto: str) -> str:
    soup = BeautifulSoup(html_bruto, "lxml")
    return soup.get_text("\n", strip=True)


AMOSTRA = 5
detalhes = []

for item in todos_itens[:AMOSTRA]:
    print(f"\n  [item] {item.get('title', '?')[:80]}")
    detalhe = obter_detalhe(item["slug"])
    if not detalhe:
        continue

    texto_limpo = limpar_html_content(detalhe.get("content", ""))
    print(f"    -> {len(texto_limpo)} chars de texto limpo.")
    print(f"    -> journal={detalhe.get('journal')!r}, section={detalhe.get('section')!r}")

    detalhes.append({
        "titulo": detalhe.get("title"),
        "data": detalhe.get("date"),
        "slug": item["slug"],
        "texto": texto_limpo,
        "publicationType": detalhe.get("publicationType"),
    })

    time.sleep(random.uniform(0.3, 0.8))

print(f"\n{len(detalhes)}/{AMOSTRA} detalhes obtidos com sucesso.")


In [0]:
d = detalhes[0]

print("=" * 100)
print(f"TÍTULO : {d['titulo']}")
print(f"DATA   : {d['data']}")
print(f"TIPO   : {d['publicationType']}")
print(f"URL    : https://doe.sp.gov.br/{d['slug']}")
print(f"TAMANHO: {len(d['texto'])} chars")
print("=" * 100)
print(d["texto"])


## Conclusão da Fase 1

Confirmar: (a) `sem_data`/`sem_slug` vazios -- sem risco do tipo 422 do
PSR; (b) texto limpo saiu legível, sem lixo de HTML/entidade sobrando.

**Decisão de arquitetura, diferente de tudo que já existe**: essa fonte não
é RSS, não é "site inteiro", não é scraping de HTML nem lista de PDF -- é
busca por termo via API JSON, mais parecida com o notebook original de
Google News (mas com API limpa em vez de RSS+decodificação de link).

Duas opções pra Fase 2:
- **Notebook próprio**, parecido com o antigo Google News, mas com API
  estruturada -- mais simples de manter que scraping.
- **Novo `modo` no Dispatcher 3** (`modo_download="api_json"`), reaproveitando
  o esqueleto de manifesto/salvamento, trocando só a etapa de download.

Termo de busca pode virar lista (`TERMOS`) -- útil se quiser monitorar mais
de uma secretaria/agência pelo mesmo mecanismo, sem multiplicar notebook.